# Tutorial X: Home Attribution

In [2]:
import pandas as pd
import pygeohash as pgh
import nomad.io.base as loader
import nomad.constants as constants
import nomad.stop_detection.ta_dbscan as DBSCAN
import nomad.stop_detection.lachesis as Lachesis
import nomad.filters as filters
import nomad.city_gen as cg

In [7]:
traj_cols = {'start_datetime': 'start_time',
            'end_datetime': 'end_time'}

stop_table = loader.from_file("../nomad/data/gc_stops.csv", traj_cols = traj_cols)

In [9]:
stop_table.head()

,start_time,end_time,longitude,latitude,diameter,n_pings,duration
0,2024-01-01 15:04:00,2024-01-01 16:16:00,-36.666470,38.320064,14.691535,14,72
1,2024-01-01 18:17:00,2024-01-01 20:58:00,-36.667398,38.320415,12.974932,19,161
2,2024-01-01 21:15:00,2024-01-02 01:10:00,-36.667525,38.321253,13.946715,32,235
3,2024-01-02 05:54:00,2024-01-02 08:01:00,-36.667489,38.321273,31.460130,17,127
4,2024-01-02 08:40:00,2024-01-02 11:59:00,-36.667721,38.320554,20.736412,39,199


In [13]:
def home_detection(stop_table, night_start, night_end, geohash_precision, min_pings, local_tz = 'America/New_York'):
    """
    Detects home locations based on stops occurring at night.

    Parameters
    ----------
    stop_table : pd.DataFrame
        DataFrame containing stop information with 'start_time' and 'end_time'.
    night_start : int
        Start hour for night period (e.g., 19 for 7:00 PM).
    night_end : int
        End hour for night period (e.g., 7 for 7:00 AM).
    geohash_precision : int
        Precision level for geohashing locations.
    min_pings : int
        Minimum number of pings to consider for a home.
    local_tz : str
        Timezone to convert timestamps (e.g., 'America/New_York').

    Returns
    -------
    pd.DataFrame
        Filtered stops within the night period with geohash encoding.
    """
    home_table = stop_table.copy()
    
    # Convert timestamps to local timezone
    home_table['start_time'] = pd.to_datetime(home_table['start_time']).dt.tz_localize('UTC').dt.tz_convert(local_tz)
    home_table['end_time'] = pd.to_datetime(home_table['end_time']).dt.tz_localize('UTC').dt.tz_convert(local_tz)

    # Stops between night_start and 24:00 (midnight)
    df1 = home_table[(home_table['start_time'].dt.hour.isin(range(night_start, 24))) & 
                     (home_table['end_time'].dt.hour.isin(range(night_start, 24)))]

    # Stops spanning night_start to night_end
    df2 = home_table[(home_table['start_time'].dt.hour.isin(range(night_start, 24))) & 
                     (home_table['end_time'].dt.hour < night_end)]

    # Stops between 00:00 and night_end
    df3 = home_table[(home_table['start_time'].dt.hour < night_end) & 
                     (home_table['end_time'].dt.hour < night_end)]

    # Combine results
    filtered_df = pd.concat([df1, df2, df3], ignore_index=True)

    filtered_df = filtered_df[filtered_df['n_pings'] >= min_pings]

    # Compute geohash
    filtered_df['pgh_encoding'] = filtered_df.apply(lambda x: pgh.encode(latitude=x['latitude'], longitude=x['longitude'], precision=geohash_precision), axis=1)

    return filtered_df

In [18]:
home_detection(stop_table, night_start = 22, night_end = 6, min_pings = 15, geohash_precision = 5)

,start_time,end_time,longitude,latitude,diameter,n_pings,duration,pgh_encoding
0,2024-01-03 23:54:00-05:00,2024-01-04 02:57:00-05:00,-36.667502,38.321389,16.362749,31,183,envfj
1,2024-01-09 22:33:00-05:00,2024-01-10 01:23:00-05:00,-36.667400,38.320404,15.939312,27,170,envfj
2,2024-01-02 00:54:00-05:00,2024-01-02 03:01:00-05:00,-36.667489,38.321273,31.460130,17,127,envfj
4,2024-01-05 03:05:00-05:00,2024-01-05 05:47:00-05:00,-36.667811,38.320681,32.546320,17,162,envfj
6,2024-01-08 03:22:00-05:00,2024-01-08 05:03:00-05:00,-36.667845,38.320605,41.718132,16,101,envfj
7,2024-01-12 01:24:00-05:00,2024-01-12 02:59:00-05:00,-36.667472,38.320984,10.926358,15,95,envfj
8,2024-01-12 03:02:00-05:00,2024-01-12 05:40:00-05:00,-36.666798,38.321656,42.543583,22,158,envfj
